# A1 — Vision Transformers: Attention, ViT do zero e Transfer Learning

**Projeto:** EcoVision AI  
**Dataset:** RealWaste  
**Objetivo:** implementar os componentes centrais de um Transformer para visão, construir um Vision Transformer (ViT) do zero, analisar seus mapas de atenção e comparar seu desempenho com um ViT pré-treinado.

## Requisitos de execução

- **Ambiente recomendado:** Google Colab com GPU **T4**
- **GPU:** T4 (16 GB de VRAM)
- **RAM recomendada:** 8 GB ou mais
- **Espaço em disco recomendado:** pelo menos 2 GB livres
- **Tempo estimado com checkpoints já existentes:** ~3–8 minutos
- **Tempo estimado treinando os dois modelos do zero:** ~20–40 minutos em T4, podendo variar conforme o ambiente
- **Seed utilizada:** 42
- **Imagem de entrada:** 224 × 224
- **Número de classes:** 9

> O notebook reutiliza os módulos do projeto em `src/vit/`.  
> Se os checkpoints não existirem, os treinamentos serão executados automaticamente.  
> Se já existirem, eles serão reutilizados para evitar treinamento desnecessário.

## 1. Preparação do ambiente

O notebook procura automaticamente a raiz do projeto. Em execução local, basta abrir o notebook dentro do projeto. No Colab, o projeto pode estar em `/content` ou no Google Drive.

Dependências adicionais utilizadas nesta atividade:

- `torch` / `torchvision`
- `timm`
- `scikit-learn`
- `matplotlib`
- `pandas`
- `Pillow`

In [ ]:
# Instala somente dependências que podem não existir no Colab.
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "timm",
        "scikit-learn",
        "matplotlib",
        "pandas",
        "pillow",
    ],
    check=True,
)

print("Dependências verificadas.")

In [ ]:
from pathlib import Path
import os
import sys

def encontrar_raiz_projeto():
    candidatos = []

    cwd = Path.cwd().resolve()
    candidatos.extend([cwd, *cwd.parents])

    candidatos.extend([
        Path("/content/EcoVision-AI-organized"),
        Path("/content/EcoVision-AI"),
    ])

    for candidato in candidatos:
        if (
            (candidato / "src" / "vit").exists()
            and (candidato / "src" / "common").exists()
        ):
            return candidato

    # Se estivermos no Colab e o projeto estiver no Google Drive,
    # montamos o Drive e fazemos uma busca curta.
    try:
        from google.colab import drive
        drive.mount("/content/drive")

        mydrive = Path("/content/drive/MyDrive")
        if mydrive.exists():
            nomes_preferidos = [
                "EcoVision-AI-organized",
                "EcoVision-AI",
            ]

            for nome in nomes_preferidos:
                encontrados = list(mydrive.glob(f"**/{nome}"))
                for candidato in encontrados:
                    if (
                        (candidato / "src" / "vit").exists()
                        and (candidato / "src" / "common").exists()
                    ):
                        return candidato.resolve()
    except Exception:
        pass

    raise FileNotFoundError(
        "Não encontrei a raiz do projeto. "
        "No Colab, coloque a pasta completa EcoVision-AI no /content "
        "ou no Google Drive."
    )

PROJECT_ROOT = encontrar_raiz_projeto()
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Raiz do projeto:", PROJECT_ROOT)
print("Diretório atual:", Path.cwd())

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Dispositivo:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM: {total_vram:.2f} GB")
else:
    print(
        "Aviso: o notebook funciona em CPU, mas os treinamentos "
        "ficarão significativamente mais lentos."
    )

## 2. Dataset RealWaste

O RealWaste é utilizado para a classificação de resíduos em nove categorias:

1. Cardboard
2. Food Organics
3. Glass
4. Metal
5. Miscellaneous Trash
6. Paper
7. Plastic
8. Textile Trash
9. Vegetation

O pipeline divide o dataset de forma **estratificada** em:

- 70% treino
- 15% validação
- 15% teste

A estratificação mantém aproximadamente a mesma proporção de classes nos três conjuntos.

### Data augmentation

No treino são utilizadas pequenas transformações para aumentar a variabilidade das imagens:

- `RandomResizedCrop`
- `RandomHorizontalFlip`
- `RandomRotation`
- `ColorJitter`

Validação e teste não recebem augmentation aleatória. Todos os conjuntos utilizam a normalização do ImageNet.

In [ ]:
from src.vit.download_dataset import (
    criar_diretorios,
    baixar_dataset,
    extrair_dataset,
    localizar_dataset as localizar_dataset_download,
)

criar_diretorios()
baixar_dataset()

try:
    dataset_path = localizar_dataset_download()
except FileNotFoundError:
    extrair_dataset()
    dataset_path = localizar_dataset_download()

print("Dataset localizado em:")
print(dataset_path)

In [ ]:
from collections import Counter
from src.vit.data_pipeline import (
    localizar_dataset,
    criar_datasets,
    criar_dataloaders,
)

dataset_path = localizar_dataset()

dataset_base, train_dataset, val_dataset, test_dataset = criar_datasets(
    dataset_path
)

train_loader, val_loader, test_loader = criar_dataloaders(
    train_dataset,
    val_dataset,
    test_dataset,
)

print("Classes:", dataset_base.class_to_idx)
print()
print("Total:", len(dataset_base))
print("Treino:", len(train_dataset))
print("Validação:", len(val_dataset))
print("Teste:", len(test_dataset))

targets = dataset_base.targets
contagem = Counter(targets)

print("\nDistribuição total:")
for idx, classe in enumerate(dataset_base.classes):
    print(f"{classe:24s}: {contagem[idx]}")

## 3. Scaled Dot-Product Attention

O mecanismo central de atenção utilizado é:

\[
Attention(Q,K,V) =
softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

### Intuição

- **Query (Q):** o que um token está procurando.
- **Key (K):** o que cada token oferece para comparação.
- **Value (V):** a informação que será combinada.
- `QKᵀ` mede a compatibilidade entre tokens.
- a divisão por `√d_k` evita scores excessivamente grandes.
- `softmax` transforma os scores em pesos.
- os pesos são aplicados aos `Values`.

No projeto, esse mecanismo foi implementado manualmente em PyTorch.

In [ ]:
import inspect
from src.vit.models.attention import (
    ScaledDotProductAttention,
    MultiHeadAttention,
)

print(inspect.getsource(ScaledDotProductAttention))

In [ ]:
# Teste isolado do Scaled Dot-Product Attention.

batch_size = 2
num_heads = 3
num_tokens = 5
head_dim = 64

query = torch.randn(batch_size, num_heads, num_tokens, head_dim)
key = torch.randn(batch_size, num_heads, num_tokens, head_dim)
value = torch.randn(batch_size, num_heads, num_tokens, head_dim)

attention = ScaledDotProductAttention()

output, attention_weights = attention(
    query,
    key,
    value,
)

print("Q:", query.shape)
print("Saída:", output.shape)
print("Pesos de atenção:", attention_weights.shape)

assert output.shape == (2, 3, 5, 64)
assert attention_weights.shape == (2, 3, 5, 5)
assert not torch.isnan(output).any()

print("Scaled Dot-Product Attention: OK")

## 4. Multi-Head Attention

Uma única cabeça de atenção aprende uma forma de relacionar os tokens. Em **Multi-Head Attention**, várias cabeças aprendem relações diferentes em paralelo.

Neste projeto:

- `embed_dim = 192`
- `num_heads = 3`
- `head_dim = 192 / 3 = 64`

Cada cabeça possui projeções independentes para **Q, K e V**. Depois:

1. cada head calcula sua própria atenção;
2. as saídas são concatenadas;
3. uma projeção linear final combina as informações.

Isso permite que diferentes heads aprendam diferentes tipos de relação espacial entre os patches.

In [ ]:
print(inspect.getsource(MultiHeadAttention))

In [ ]:
mha = MultiHeadAttention(
    embed_dim=192,
    num_heads=3,
    dropout=0.0,
)

x = torch.randn(2, 197, 192)

mha_output, mha_weights = mha(x)

print("Entrada:", x.shape)
print("Saída:", mha_output.shape)
print("Attention weights:", mha_weights.shape)

assert mha_output.shape == (2, 197, 192)
assert mha_weights.shape == (2, 3, 197, 197)

print("Multi-Head Attention: OK")

## 5. Transformer Encoder Block

O bloco Encoder utilizado no EcoVision segue uma arquitetura **Pre-Norm**:

```text
x
│
LayerNorm
│
Multi-Head Attention
│
+ residual
▼
x
│
LayerNorm
│
Feed Forward
│
+ residual
▼
output
```

### Componentes

**LayerNorm** estabiliza a representação dos tokens.

**Residual connection** soma a entrada original à transformação:

\[
y = x + F(x)
\]

Isso facilita o fluxo de gradientes em redes profundas.

**Feed Forward Network (FFN)** processa cada token individualmente:

```text
192
 ↓
Linear
 ↓
768
 ↓
GELU
 ↓
Dropout
 ↓
Linear
 ↓
192
```

In [ ]:
from src.vit.models.transformer import (
    FeedForward,
    TransformerEncoderBlock,
)

print(inspect.getsource(TransformerEncoderBlock))

In [ ]:
encoder = TransformerEncoderBlock(
    embed_dim=192,
    num_heads=3,
    mlp_dim=768,
    dropout=0.1,
)

encoder_output, encoder_attention = encoder(
    torch.randn(2, 197, 192)
)

print("Saída:", encoder_output.shape)
print("Attention:", encoder_attention.shape)

assert encoder_output.shape == (2, 197, 192)
assert encoder_attention.shape == (2, 3, 197, 197)

print("Transformer Encoder Block: OK")

## 6. Como o ViT transforma uma imagem em tokens

Uma imagem de `224 × 224` é dividida em patches de `16 × 16`.

\[
224 / 16 = 14
\]

Portanto:

\[
14 \times 14 = 196\ patches
\]

Cada patch é convertido para um embedding de dimensão `192`.

Em seguida são adicionados:

- um **CLS Token** treinável;
- um **Positional Embedding** treinável.

Resultado:

```text
196 patches
+ 1 CLS token
= 197 tokens

Shape:
[batch, 197, 192]
```

### Por que o Positional Embedding é necessário?

A operação de self-attention não conhece, por si só, a posição original dos patches. Sem informação posicional, o modelo saberia quais tokens existem, mas não onde cada região estava localizada na imagem.

O positional embedding injeta essa informação espacial na representação.

In [ ]:
from src.vit.models.vit import (
    PatchEmbedding,
    ViTEmbeddings,
)

print(inspect.getsource(PatchEmbedding))
print(inspect.getsource(ViTEmbeddings))

In [ ]:
imagem_fake = torch.randn(2, 3, 224, 224)

patch_embedding = PatchEmbedding(
    image_size=224,
    patch_size=16,
    in_channels=3,
    embed_dim=192,
)

patches = patch_embedding(imagem_fake)

embeddings = ViTEmbeddings(
    image_size=224,
    patch_size=16,
    in_channels=3,
    embed_dim=192,
    dropout=0.1,
)

tokens = embeddings(imagem_fake)

print("Imagem:", imagem_fake.shape)
print("Patch embeddings:", patches.shape)
print("CLS + patches + posição:", tokens.shape)

assert patches.shape == (2, 196, 192)
assert tokens.shape == (2, 197, 192)

print("Patch/ViT Embeddings: OK")

## 7. Vision Transformer completo implementado do zero

A arquitetura utilizada no experimento foi:

| Parâmetro | Valor |
|---|---:|
| Image size | 224 × 224 |
| Patch size | 16 × 16 |
| Patches | 196 |
| Tokens com CLS | 197 |
| Embedding | 192 |
| Attention heads | 3 |
| Dimensão de cada head | 64 |
| Encoder blocks | 4 |
| FFN / MLP dimension | 768 |
| Dropout | 0.1 |
| Classes | 9 |

Fluxo completo:

```text
Imagem
 ↓
Patch Embedding
 ↓
CLS Token + Positional Embedding
 ↓
4 Transformer Encoder Blocks
 ↓
LayerNorm
 ↓
CLS Token final
 ↓
Linear 192 → 9
 ↓
Logits
```

In [ ]:
from src.vit.models.vit_model import EcoVisionViT

print(inspect.getsource(EcoVisionViT))

In [ ]:
vit_scratch = EcoVisionViT(
    image_size=224,
    patch_size=16,
    num_classes=9,
    embed_dim=192,
    num_heads=3,
    num_layers=4,
    mlp_dim=768,
    dropout=0.1,
)

entrada = torch.randn(2, 3, 224, 224)

logits, attentions = vit_scratch(
    entrada,
    return_attention=True,
)

num_params = sum(p.numel() for p in vit_scratch.parameters())

print("Logits:", logits.shape)
print("Número de Encoder Blocks:", len(attentions))
print("Attention último bloco:", attentions[-1].shape)
print(f"Parâmetros: {num_params:,}")

assert logits.shape == (2, 9)
assert len(attentions) == 4
assert attentions[-1].shape == (2, 3, 197, 197)

print("EcoVision ViT completo: OK")

## 8. Treinamento do ViT do zero

Configuração utilizada:

- **Loss:** CrossEntropyLoss
- **Optimizer:** AdamW
- **Learning rate:** `3e-4`
- **Weight decay:** `0.05`
- **Máximo:** 15 épocas
- **Early stopping patience:** 4

### Forward, loss e backward

Durante cada batch:

```text
imagem
 ↓
forward
 ↓
logits
 ↓
CrossEntropyLoss(logits, classe_real)
 ↓
loss.backward()
 ↓
gradientes
 ↓
optimizer.step()
 ↓
pesos atualizados
```

O melhor checkpoint é escolhido utilizando a perda de validação.

In [ ]:
import subprocess
import sys

scratch_ckpt = PROJECT_ROOT / "models" / "ecovision_vit_scratch_best.pth"

if scratch_ckpt.exists():
    print("Checkpoint do ViT scratch já existe. Treinamento será reutilizado:")
    print(scratch_ckpt)
else:
    print("Checkpoint não encontrado. Iniciando treinamento do ViT do zero...")
    subprocess.run(
        [sys.executable, "-m", "src.vit.train_scratch"],
        cwd=PROJECT_ROOT,
        check=True,
    )

In [ ]:
# Avaliação reproduzível no conjunto de teste.
subprocess.run(
    [sys.executable, "-m", "src.vit.evaluate_scratch"],
    cwd=PROJECT_ROOT,
    check=True,
)

In [ ]:
import json
import pandas as pd

from sklearn.metrics import f1_score
from src.vit.evaluate_scratch import (
    carregar_modelo as carregar_vit_scratch,
    obter_predicoes as obter_predicoes_scratch,
)

scratch_results_path = PROJECT_ROOT / "results" / "vit_scratch_test_results.json"

with open(scratch_results_path, "r", encoding="utf-8") as f:
    scratch_results = json.load(f)

# O JSON histórico do ViT scratch armazena as métricas weighted,
# mas não salvava o Macro F1. Para manter o notebook reproduzível,
# recalculamos essa métrica sobre o mesmo conjunto de teste.
modelo_scratch_avaliacao = carregar_vit_scratch()

y_true_scratch, y_pred_scratch = obter_predicoes_scratch(
    modelo_scratch_avaliacao,
    test_loader,
)

scratch_f1_macro = f1_score(
    y_true_scratch,
    y_pred_scratch,
    average="macro",
    zero_division=0,
)

resumo_scratch = pd.DataFrame({
    "Métrica": [
        "Accuracy",
        "Precision weighted",
        "Recall weighted",
        "F1 weighted",
        "F1 macro",
    ],
    "Valor": [
        scratch_results["accuracy"],
        scratch_results["precision_weighted"],
        scratch_results["recall_weighted"],
        scratch_results["f1_weighted"],
        scratch_f1_macro,
    ],
})

resumo_scratch["Valor (%)"] = resumo_scratch["Valor"] * 100

resumo_scratch[["Métrica", "Valor (%)"]]

### Resultado validado — ViT do zero

No experimento realizado:

- **melhor época:** 13
- **validation accuracy:** 58,49%
- **test accuracy:** **55,26%**
- **weighted F1:** **53,67%**
- **macro F1:** **52,44%**

O resultado é coerente com uma característica importante dos Transformers visuais: **ViTs normalmente são mais dependentes de quantidade de dados e pré-treinamento do que CNNs**. Com apenas 4.752 imagens, nosso modelo precisa aprender simultaneamente representações visuais e a tarefa de classificação.

## 9. Attention Maps

Os attention maps ajudam a inspecionar **quais patches receberam maior peso de atenção**.

No projeto utilizamos a atenção do **CLS Token** em relação aos patches do último Transformer Encoder.

Para cada head:

```text
attention[CLS → patches]
 ↓
196 valores
 ↓
reshape 14 × 14
 ↓
interpolação 224 × 224
 ↓
sobreposição na imagem
```

Esses mapas ajudam na interpretação do comportamento interno do modelo, porém **atenção não deve ser tratada automaticamente como explicação causal** da decisão.

In [ ]:
# Gera/atualiza os mapas de atenção utilizando o checkpoint scratch.
subprocess.run(
    [sys.executable, "-m", "src.vit.attention_maps"],
    cwd=PROJECT_ROOT,
    check=True,
)

In [ ]:
from IPython.display import display, Image as IPyImage

attention_dir = PROJECT_ROOT / "results" / "attention_maps"

arquivos = [
    attention_dir / "attention_original.png",
    attention_dir / "attention_overlay_head_0.png",
    attention_dir / "attention_overlay_head_1.png",
    attention_dir / "attention_overlay_head_2.png",
]

for arquivo in arquivos:
    if arquivo.exists():
        print(arquivo.name)
        display(IPyImage(filename=str(arquivo)))

### Interpretação dos heads

Os heads não precisam olhar para as mesmas regiões. Essa é justamente uma das vantagens do Multi-Head Attention: cada cabeça pode aprender relações diferentes entre os patches.

No experimento do EcoVision, os mapas apresentaram padrões distintos entre os três heads. Isso fornece evidência de que diferentes projeções de Q/K/V aprenderam diferentes relações espaciais.

A interpretação deve ser feita com cautela: um mapa de atenção indica distribuição de atenção, mas não prova sozinho que uma região foi causalmente responsável pela classificação.

## 10. ViT pré-treinado — Transfer Learning

A segunda abordagem usa:

```text
vit_tiny_patch16_224
```

pré-treinado e disponibilizado pela biblioteca `timm`.

O classification head original é substituído:

```text
192 features
 ↓
Linear
 ↓
9 classes RealWaste
```

O treinamento ocorre em duas fases.

### Fase 1 — Feature Extraction

Todo o backbone é congelado. Apenas o novo classification head é treinado.

```text
Backbone pré-treinado → congelado
Classification head  → treinável
```

Isso preserva as representações visuais já aprendidas.

### Fase 2 — Fine-Tuning

Depois, todo o modelo é descongelado e treinado com learning rate menor:

```text
HEAD:
2 epochs
LR = 1e-3

FINE-TUNING:
8 epochs
LR = 1e-5
```

O learning rate menor no fine-tuning reduz o risco de destruir rapidamente os pesos pré-treinados.

In [ ]:
from src.vit.models.pretrained_vit import (
    EcoVisionPretrainedViT,
    contar_parametros,
)

modelo_pre = EcoVisionPretrainedViT(
    num_classes=9,
    pretrained=True,
)

total, treinaveis = contar_parametros(modelo_pre)
print(f"Total de parâmetros: {total:,}")
print(f"Treináveis inicialmente: {treinaveis:,}")

modelo_pre.congelar_backbone()

_, treinaveis_congelado = contar_parametros(modelo_pre)

print(f"Treináveis com backbone congelado: {treinaveis_congelado:,}")
assert treinaveis_congelado == 1737

modelo_pre.descongelar_modelo()

_, treinaveis_finetune = contar_parametros(modelo_pre)

print(f"Treináveis no fine-tuning: {treinaveis_finetune:,}")

In [ ]:
pretrained_ckpt = PROJECT_ROOT / "models" / "ecovision_vit_pretrained_best.pth"

if pretrained_ckpt.exists():
    print("Checkpoint do ViT pré-treinado já existe. Treinamento será reutilizado:")
    print(pretrained_ckpt)
else:
    print("Checkpoint não encontrado. Iniciando feature extraction + fine-tuning...")
    subprocess.run(
        [sys.executable, "-m", "src.vit.train_pretrained"],
        cwd=PROJECT_ROOT,
        check=True,
    )

In [ ]:
subprocess.run(
    [sys.executable, "-m", "src.vit.evaluate_pretrained"],
    cwd=PROJECT_ROOT,
    check=True,
)

In [ ]:
pretrained_results_path = PROJECT_ROOT / "results" / "vit_pretrained_test_results.json"

with open(pretrained_results_path, "r", encoding="utf-8") as f:
    pretrained_results = json.load(f)

pretrained_test = pretrained_results["test"]

comparacao = pd.DataFrame({
    "Modelo": [
        "ViT do zero",
        "ViT pré-treinado",
    ],
    "Accuracy (%)": [
        scratch_results["accuracy"] * 100,
        pretrained_test["accuracy"] * 100,
    ],
    "F1 weighted (%)": [
        scratch_results["f1_weighted"] * 100,
        pretrained_test["f1_weighted"] * 100,
    ],
    "F1 macro (%)": [
        scratch_f1_macro * 100,
        pretrained_test["f1_macro"] * 100,
    ],
})

comparacao

In [ ]:
ganho_accuracy = (
    pretrained_test["accuracy"]
    - scratch_results["accuracy"]
) * 100

print(
    f"Ganho de accuracy do ViT pré-treinado em relação "
    f"ao ViT do zero: {ganho_accuracy:.2f} pontos percentuais."
)

### Resultado validado — comparação quantitativa

| Modelo | Test Accuracy | Weighted F1 | Macro F1 |
|---|---:|---:|---:|
| ViT do zero | **55,26%** | **53,67%** | **52,44%** |
| ViT pré-treinado | **86,54%** | **86,55%** | **87,24%** |

O ganho em accuracy foi de **31,28 pontos percentuais**.

A comparação mostra o impacto do **transfer learning** neste dataset. O ViT do zero precisa aprender representações visuais apenas com o RealWaste. Já o modelo pré-treinado começa com representações previamente aprendidas e precisa adaptá-las ao domínio de resíduos.

## 11. BERT × ViT

BERT e ViT utilizam o **Transformer Encoder**, mas operam sobre modalidades diferentes.

| Aspecto | BERT | ViT |
|---|---|---|
| Entrada | tokens de texto | patches de imagem |
| Embedding | token embedding | patch embedding |
| Token especial | `[CLS]` | CLS token |
| Informação de posição | positional embedding | positional embedding |
| Núcleo | Transformer Encoder | Transformer Encoder |
| Saída de classificação | representação do `[CLS]` | representação do CLS |

### Objetivos de pré-treinamento

No BERT original, o principal objetivo é **Masked Language Modeling (MLM)**: alguns tokens são ocultados e o modelo aprende a reconstruí-los. O trabalho original também utiliza **Next Sentence Prediction (NSP)**.

O ViT original foi apresentado principalmente utilizando **pré-treinamento supervisionado para classificação de imagens**, seguido de adaptação/fine-tuning para outros datasets.

Assim, a arquitetura Transformer é reutilizada, mas a forma de construir a entrada e o objetivo de pré-treinamento são diferentes.

## 12. DeiT — Data-efficient Image Transformer

Uma dificuldade dos primeiros ViTs era a forte dependência de grandes volumes de dados.

O **DeiT (Data-efficient Image Transformer)** foi proposto para tornar o treinamento de Transformers visuais mais eficiente em dados.

Um elemento importante é a **distillation token**. Além do CLS token, o modelo pode receber um token dedicado à destilação de conhecimento de um modelo professor.

Conceitualmente:

```text
CNN / professor
       ↓
conhecimento
       ↓
DeiT / aluno
```

Isso permite transferir conhecimento de um modelo já forte para o Transformer durante o treinamento.

### Relação com nosso experimento

Nosso ViT scratch teve 55,26% de accuracy, enquanto o ViT pré-treinado atingiu 86,54%. Esse resultado reforça, no contexto específico deste experimento, a importância de representações previamente aprendidas quando o conjunto de dados é relativamente pequeno.

## 13. Swin Transformer

O ViT padrão calcula self-attention global entre todos os tokens. À medida que o número de patches cresce, essa operação pode se tornar cara.

O **Swin Transformer** organiza a atenção de forma hierárquica e utiliza janelas locais.

```text
imagem
 ↓
patches
 ↓
atenção em janelas locais
 ↓
shifted windows
 ↓
representações hierárquicas
```

O mecanismo de **shifted windows** desloca as janelas entre blocos para permitir comunicação entre regiões que anteriormente estavam separadas.

### O que o Swin tenta resolver

- reduzir o custo da atenção global;
- construir representações hierárquicas;
- facilitar o uso de Transformers em tarefas densas, como detecção e segmentação.

## 14. CNN × Vision Transformer

### CNN

Uma CNN possui forte **viés indutivo local**:

- convoluções observam regiões locais;
- compartilhamento de pesos;
- hierarquia espacial construída progressivamente.

Isso costuma ser vantajoso quando há menos dados.

### ViT

O ViT transforma regiões da imagem em tokens e utiliza self-attention.

Vantagens:

- relações globais entre patches podem ser modeladas desde os primeiros blocos;
- arquitetura altamente compatível com pré-treinamento em larga escala.

Desvantagem importante:

- quando treinado do zero, tende a depender mais de quantidade de dados e regularização.

### Evidência no EcoVision

Neste projeto:

```text
ViT do zero          55,26%
ViT pré-treinado     86,54%
```

A diferença não demonstra que "pré-treinado sempre vence", mas mostra claramente que, **neste dataset e configuração**, o pré-treinamento foi decisivo para a performance do ViT.

## 15. Justificativa da arquitetura utilizada

O ViT scratch foi deliberadamente construído em escala reduzida:

- embedding de 192;
- 3 heads;
- 4 Encoder Blocks;
- MLP de 768;
- patch 16 × 16.

Essa configuração mantém todos os componentes fundamentais de um ViT, mas reduz custo computacional e número de parâmetros, tornando possível treinar e estudar a arquitetura em uma máquina sem GPU e posteriormente reproduzi-la no Colab T4.

O objetivo da implementação do zero não era superar modelos industriais, e sim demonstrar corretamente:

- patch embedding;
- CLS token;
- positional embedding;
- scaled dot-product attention;
- Multi-Head Attention;
- Feed Forward;
- LayerNorm;
- residual connections;
- classificação pelo CLS token.

## 16. Conclusões da Atividade 1

Os experimentos permitiram observar três pontos principais.

**1. O ViT do zero funcionou, mas teve desempenho limitado.**  
Ele atingiu 55,26% de accuracy no conjunto de teste.

**2. O transfer learning produziu grande ganho.**  
O ViT pré-treinado atingiu 86,54%, uma diferença de 31,28 pontos percentuais.

**3. Attention Maps permitem inspecionar relações aprendidas pelos heads.**  
Os heads apresentaram padrões distintos, embora mapas de atenção não devam ser interpretados como prova causal da decisão.

### Resumo para estudo

```text
Imagem
→ patches
→ embeddings
→ + CLS
→ + posição
→ Multi-Head Self-Attention
→ Feed Forward
→ residual + LayerNorm
→ repetir Encoder
→ CLS final
→ classificação
```

A principal mensagem experimental é:

```text
ViT treinado do zero
       ↓
depende fortemente dos dados disponíveis

ViT pré-treinado
       ↓
reutiliza representações visuais
       ↓
melhor adaptação ao RealWaste
```

## 17. Checklist da rubrica — Atividade 1

- [x] Scaled Dot-Product Attention implementado do zero
- [x] Multi-Head Attention com projeções independentes por head e concatenação
- [x] Transformer Encoder Block com FFN, LayerNorm e residual connections
- [x] Patch Embedding
- [x] CLS Token aprendível
- [x] Positional Embedding
- [x] Fluxo completo imagem → logits
- [x] ViT treinado do zero
- [x] Attention Maps e interpretação
- [x] ViT pré-treinado
- [x] Feature extraction
- [x] Fine-tuning
- [x] Comparação quantitativa scratch × pré-treinado
- [x] Explicação BERT × ViT
- [x] Explicação de positional embeddings
- [x] Análise de DeiT
- [x] Análise de Swin Transformer
- [x] Discussão ViT × CNN
- [x] Justificativa da arquitetura utilizada